<a href="https://colab.research.google.com/github/sj-workbench/years_of_learnings/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [104]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [105]:
df = pd.read_csv('/content/Titanic-Dataset.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [106]:
df.drop(columns = ['PassengerId','Name','Ticket','Cabin'], inplace = True)

In [107]:
x = df.drop('Survived', axis = 1)
y = df['Survived']

In [108]:
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)

In [109]:
x_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [110]:
# Simple imputation transformer
trf1 = ColumnTransformer(transformers = [
    ('impute_age', SimpleImputer(),[2]),
    ('impute_embarked', SimpleImputer(strategy = 'most_frequent'), [6])
    ], remainder = 'passthrough')

In [111]:
# One hot encoding
trf2 = ColumnTransformer(transformers = [
    ('ohe_sex_embarked', OneHotEncoder(sparse_output = False, handle_unknown= 'ignore'),[3,1])
    ], remainder = 'passthrough')

In [112]:
# Scaling
trf3 = ColumnTransformer(transformers = [('scale', MinMaxScaler(), slice(0,10))], remainder = 'passthrough')

In [113]:
# Feature Selection
trf4 = SelectKBest(score_func = chi2, k=8)

In [114]:
# train the model
trf5 = DecisionTreeClassifier()

In [115]:
# Making a Pipeline
pipe = Pipeline([('trf1', trf1),('trf2', trf2),('trf3', trf3),('trf4',trf4),('trf5', trf5)])

In [116]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [3, 1])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x79266f4ab7e0>)),
                ('trf5', DecisionTreeClassifier())])

In [117]:
# predict
y_pred = pipe.predict(x_test)

In [118]:
accuracy_score(y_test, y_pred)

0.7877094972067039